Objetivo

- Aproximar ΔΔG de mutaciones puntuales como diferencia de energías electrónicas entre “wild-type” y “mutante” usando moléculas proxy de las cadenas laterales.
- Calcular energías con VQE ( UCCSD , base STO-3G ) y convertir a kcal/mol . Opcional: reducir el tamaño con “active space” para usar recursos cuánticos reales.

Setup

- Entorno:
  - pip install qiskit qiskit-aer qiskit-nature pyscf
  - Opcional IBM Quantum: pip install qiskit-ibm-runtime
- Cuenta IBM Quantum:
  - Crea cuenta y recupera tu token en https://quantum.cloud.ibm.com/ [0].
  - Elimina claves del código, usa variables de entorno.
Proxy por mutación

- 1BTL E104K (GLU → LYS): ácido acético ( CH3COOH ) vs metilamina ( CH3NH2 )
- 1RX2 F98Y (PHE → TYR): benceno ( C6H6 ) vs fenol ( C6H5OH )
- 1KZN S83L (SER → LEU): metanol ( CH3OH ) vs isobutano ( C4H10 , aproximable con butano)
- 1KZN D87N (ASP → ASN): ácido fórmico ( HCOOH ) vs formamida ( HCONH2 )
Para un MVP, empieza con 2 pares (p. ej., E104K y D87N), y después añade los otros.

Código base (local, VQE en simulador)



In [14]:
pip install -U qiskit-nature


  Using cached qiskit_nature-0.7.2-py3-none-any.whl.metadata (8.0 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached h5py-3.15.1-cp312-cp312-win_amd64.whl.metadata (3.1 kB)
Using cached qiskit_nature-0.7.2-py3-none-any.whl (2.2 MB)
Using cached setuptools-80.9.0-py3-none-any.whl (1.2 MB)
Using cached h5py-3.15.1-cp312-cp312-win_amd64.whl (2.9 MB)

   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   -

In [11]:
pip install -U qiskit-aer qiskit-ibm-runtime


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install -U pip setuptools wheel


Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pyscf-2.6.1-cp312-cp312-win_amd64.whl


Processing c:\users\max\documents\github\qteam\pyscf-2.6.1-cp312-cp312-win_amd64.whl
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\Max\\Documents\\GitHub\\QTeam\\pyscf-2.6.1-cp312-cp312-win_amd64.whl'



In [10]:
# Qiskit Nature + VQE (proxies de cadenas laterales)
import numpy as np
from qiskit_aer.primitives import Estimator as AerEstimator

from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import SLSQP, COBYLA
from qiskit.circuit.library import TwoLocal
# PySCF eliminado
# PySCF eliminado
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.operators import FermionicOp
# PySCF eliminado
# PySCF eliminado
# IBM Runtime opcional
try:
    from qiskit_ibm_runtime import QiskitRuntimeService, Estimator as RuntimeEstimator, Session
    HAVE_IBM_RUNTIME = True
except Exception:
    HAVE_IBM_RUNTIME = False

# Configuración IBM Runtime (sin os.getenv). Rellena si quieres usar IBM:
IBM_RUNTIME_TOKEN = None  # p.ej., "xxxxxxxx"
IBM_RUNTIME_INSTANCE = "ibm-q/open/main"
IBM_RUNTIME_BACKEND = None  # p.ej., "ibm_brisbane"

# PySCF eliminado: no usar rutas que dependan de PySCF

# PySCF eliminado: no se usan geometrías PySCF en esta ruta

def _make_estimator():
    # Si hay token configurado, usar IBM Runtime; si no, usar AerEstimator local
    if HAVE_IBM_RUNTIME and IBM_RUNTIME_TOKEN:
        try:
            service = QiskitRuntimeService(channel="ibm_quantum", token=IBM_RUNTIME_TOKEN, instance=IBM_RUNTIME_INSTANCE)
            backend = service.backend(IBM_RUNTIME_BACKEND) if IBM_RUNTIME_BACKEND else service.least_busy(simulator=False, operational=True)
            session = Session(service=service, backend=backend)
            return RuntimeEstimator(session=session), session
        except Exception:
            pass
    try:
        return AerEstimator(), None
    except Exception:
        raise RuntimeError("No hay Estimator disponible: instala qiskit-aer o configura IBM Runtime")

# PySCF eliminado: no hay funciones basadas en geometrías/driver químico en esta ruta

# Pares de prueba alineados con tus mutaciones
pairs = {
    "1BTL_E104K": ("acetic_acid", "methylamine"),
    "1RX2_F98Y": ("benzene", "phenol"),
    "1KZN_S83L": ("methanol", "butane"),
    "1KZN_D87N": ("formic_acid", "formamide"),
}

results = []
# PySCF eliminado: usar directamente el modelo Hubbard
print("Usando modelo Hubbard proxy (sin PySCF)")
def hubbard_ring(n_sites, t, U, onsite_eps=None):
        terms = {}
        for i in range(n_sites):
            j = (i+1) % n_sites
            for spin in [0,1]:
                terms[(i*2+spin, ">", j*2+spin, "<")] = terms.get((i*2+spin, ">", j*2+spin, "<"), 0) - t
                terms[(j*2+spin, ">", i*2+spin, "<")] = terms.get((j*2+spin, ">", i*2+spin, "<"), 0) - t
        for i in range(n_sites):
            up = i*2; dn = i*2+1
            terms[(up, "+-", dn, "+-")] = terms.get((up, "+-", dn, "+-"), 0) + U
        if onsite_eps is not None:
            for i in range(n_sites):
                eps = onsite_eps[i]
                if eps != 0:
                    for spin in [0,1]:
                        terms[(i*2+spin, "+-")] = terms.get((i*2+spin, "+-"), 0) + eps
        return FermionicOp(terms, num_spin_orbitals=n_sites*2)
def vqe_energy_fermionic(op):
    mapper = JordanWignerMapper()
    qubit_op = mapper.map(op)
    ansatz = TwoLocal(rotation_blocks="ry", entanglement_blocks="cx", reps=2, entanglement="linear")
    estimator, session = _make_estimator()
    solver = VQE(estimator, ansatz, SLSQP(maxiter=200))
    res = solver.compute_minimum_eigenvalue(qubit_op)
    if session is not None:
        try: session.close()
        except Exception: pass
    return float(np.real(res.eigenvalue))

def build_hubbard(n_sites, t, U, onsite_eps=None):
    terms = {}
    # Hopping: -t (c^†_{i,σ} c_{j,σ} + h.c.)
    for i in range(n_sites):
        j = (i + 1) % n_sites
        for spin in [0, 1]:
            p_i = i*2 + spin
            p_j = j*2 + spin
            label_ij = f"+_{p_i} -_{p_j}"
            label_ji = f"+_{p_j} -_{p_i}"
            terms[label_ij] = terms.get(label_ij, 0.0) - t
            terms[label_ji] = terms.get(label_ji, 0.0) - t
    # Onsite interaction U: n_up * n_down
    for i in range(n_sites):
        up = i*2
        dn = i*2 + 1
        label_U = f"+_{up} -_{up} +_{dn} -_{dn}"
        terms[label_U] = terms.get(label_U, 0.0) + U
    # Onsite energies ε_i
    if onsite_eps is not None:
        for i in range(n_sites):
            eps = onsite_eps[i]
            if eps != 0:
                for spin in [0, 1]:
                    p = i*2 + spin
                    label_eps = f"+_{p} -_{p}"
                    terms[label_eps] = terms.get(label_eps, 0.0) + eps
    return FermionicOp(terms, num_spin_orbitals=n_sites*2)

# proxies para cada caso (top-level)
def case_acid_amine():
    H_acid = build_hubbard(4, 1.0, 2.0, onsite_eps=[-0.6,0,0,0])
    H_amine = build_hubbard(4, 1.0, 2.0, onsite_eps=[+0.4,0,0,0])
    Ew = vqe_energy_fermionic(H_acid); Em = vqe_energy_fermionic(H_amine); d = Em-Ew
    return {"tag": "1BTL_E104K(Hubbard)", "E_wt": Ew, "E_mut": Em, "ΔE_model": d, "decision": "desestabiliza" if d>0 else "estabiliza"}
def case_benzene_phenol():
    H_bz = build_hubbard(6, 1.0, 2.0, onsite_eps=[0]*6)
    H_ph = build_hubbard(6, 1.0, 2.0, onsite_eps=[-0.5]+[0]*5)
    Ew = vqe_energy_fermionic(H_bz); Em = vqe_energy_fermionic(H_ph); d = Em-Ew
    return {"tag": "1RX2_F98Y(Hubbard)", "E_wt": Ew, "E_mut": Em, "ΔE_model": d, "decision": "desestabiliza" if d>0 else "estabiliza"}
def case_methanol_butane():
    H_me = build_hubbard(4, 1.0, 2.0, onsite_eps=[-0.6,0,0,0])
    H_bu = build_hubbard(4, 1.0, 2.0, onsite_eps=[0,0,0,0])
    Ew = vqe_energy_fermionic(H_me); Em = vqe_energy_fermionic(H_bu); d = Em-Ew
    return {"tag": "1KZN_S83L(Hubbard)", "E_wt": Ew, "E_mut": Em, "ΔE_model": d, "decision": "desestabiliza" if d>0 else "estabiliza"}
def case_formic_formamide():
    H_fa = build_hubbard(4, 1.0, 2.0, onsite_eps=[-0.5,0,0,0])
    H_fm = build_hubbard(4, 1.0, 2.0, onsite_eps=[+0.3,0,0,0])
    Ew = vqe_energy_fermionic(H_fa); Em = vqe_energy_fermionic(H_fm); d = Em-Ew
    return {"tag": "1KZN_D87N(Hubbard)", "E_wt": Ew, "E_mut": Em, "ΔE_model": d, "decision": "desestabiliza" if d>0 else "estabiliza"}

results = [case_acid_amine(), case_benzene_phenol(), case_methanol_butane(), case_formic_formamide()]

import pandas as pd
df_hubbard = pd.DataFrame(results)
display(df_hubbard)
df_hubbard.to_csv("ddg_hubbard_results.csv", index=False)

ImportError: cannot import name 'Estimator' from 'qiskit.primitives' (c:\Users\Max\Documents\GitHub\QTeam\.venv\Lib\site-packages\qiskit\primitives\__init__.py)